# Ejercicio 2

Resuelva (aproximadamente) el "Traveling salesman problem" para 200 ciudades con una red de Kohonen.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(110007)
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "Computer Modern"
})

In [ ]:
class KohonenNet:
    def __init__(self, n_neurons=10):
        self.weights = np.random.uniform(low=-1.0, high=1.0, size=(n_neurons, 2))
        self.n_neurons = n_neurons
        self.saved_weights = []
        self.saved_lengths = []

    def train(self, X, epochs=100, lr=0.01, sigma_sq_start=1.0, save_epochs=None, save_lengths=False):
        n_samples = X.shape[0]
        
        N = self.weights.shape[0]
        grid = np.arange(N)
        
        for epoch in range(epochs):
            idx = np.random.permutation(n_samples)
            sigma_sq = sigma_sq_start * (1 - epoch / epochs)

            if save_epochs and (epoch + 1) in save_epochs:
                self.saved_weights.append(np.copy(self.weights))
            if save_lengths:
                self.saved_lengths.append(self.path_length())

            for i in idx:
                x = X[i]
                dist_sq = np.sum((self.weights - x) ** 2, axis=1)
                i_star = np.argmin(dist_sq)
                abs_diff = np.abs(grid - i_star)
                dist_grid = np.minimum(abs_diff, N - abs_diff)
                dist_sq_grid = dist_grid ** 2
                neighbourhood = np.exp(-dist_sq_grid / (2 * sigma_sq))
                delta_weights = lr * neighbourhood[:, np.newaxis] * (x - self.weights)
                self.weights += delta_weights                
                
    def path_length(self):
        W = np.squeeze(self.weights) 
        W_next = np.roll(W, shift=-1, axis=0)
        dist_sq = np.sum((W - W_next) ** 2, axis=1)
        distancias = np.sqrt(dist_sq)
        length = np.sum(distancias)
        return length

In [ ]:
def visualize(epochs, weight_list, dataset, name):
    n_subplots = len(epochs)
    plt.figure(figsize=(3.1 * n_subplots, 3))

    for k in range(n_subplots):
        ax = plt.subplot(1, n_subplots, k + 1)
        
        ax.scatter(dataset[:, 0], dataset[:, 1], s=10, label='Ciudades')
    
        weights = weight_list[k]
        weights_closed = np.vstack([weights, weights[[0]]])
        ax.plot(weights_closed[:, 0], weights_closed[:, 1], 'r-', alpha=0.5, label='Camino')
        
        ax.axis('square')
        ax.set_xlim((-1.1, 1.1))
        ax.set_ylim((-1.1, 1.1))
        ax.set_xlabel('$x_1$')
        ax.set_ylabel('$x_2$')
        ax.set_title(f'Época {epochs[k]}')
        ax.legend(loc='upper right')
        ax.grid()

    plt.tight_layout()
    plt.savefig(f'../informe/img/ej2/{name}_map.svg')
    plt.show()

In [ ]:
def visualize_dataset(dataset):
    plt.figure(figsize=(3, 3))
    plt.scatter(dataset[:, 0], dataset[:, 1], s=10, label='Dataset')
    plt.xlim((-1.1, 1.1))
    plt.ylim((-1.1, 1.1))
    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
dataset = np.random.uniform(-1, 1, size=(20, 2))
visualize_dataset(dataset)

In [ ]:
save_epochs = [1, 20, 750, 1000]
model = KohonenNet(n_neurons=40)
model.train(dataset, sigma_sq_start=10.0, lr=0.04, epochs=1000, save_epochs=save_epochs, save_lengths=True)
visualize(save_epochs, model.saved_weights, dataset, '20')

In [ ]:
def visualize_lengths(lengths, name):
    plt.figure(figsize=(6, 4))
    plt.plot(lengths)
    plt.grid()
    plt.semilogy()
    plt.xlabel('Época')
    plt.ylabel('Longitud del camino')
    plt.tight_layout()
    plt.savefig(f'../informe/img/ej2/{name}_length.svg')
    plt.show()

In [ ]:
visualize_lengths(model.saved_lengths, '20')
final_length = model.saved_lengths[-1]
print(f"Final path length: {final_length:.2f}")

In [ ]:
dataset = np.random.uniform(-1, 1, size=(200, 2))
visualize_dataset(dataset)

In [ ]:
save_epochs = [1, 20, 750, 1000]
model = KohonenNet(n_neurons=400)
model.train(dataset, sigma_sq_start=100.0, lr=0.3, epochs=1000, save_epochs=save_epochs, save_lengths=True)
visualize(save_epochs, model.saved_weights, dataset, '200')

In [ ]:
visualize_lengths(model.saved_lengths, '200')
final_length = model.saved_lengths[-1]
print(f"Final path length: {final_length:.2f}")